# 06 · Lakeflow Pipeline (Spark Declarative Pipeline)

Rebuilds the cleaning flow of notebooks 01→05 as a single declarative pipeline.
The cleaning logic reuses the talaria package (`build_silver_base`, `build_silver_facilities`, `build_silver_stage3`) as-is,
and results are published to the `workspace.talaria_pipeline` schema.

Flow: source tables → bronze (3 snapshots) → silver base (objective cleaning) → silver facilities (A–H flags) → silver stage3 (rule flags) + rule log → gold (confidence scoring, spec #37) + score logs

In [0]:
import datetime
import sys

sys.path.insert(0, "/Workspace/Users/cyhun18@gmail.com/talaria")

from pyspark import pipelines as dp
from pyspark.sql import functions as F

from config import SOURCE
from talaria.silver_base import build_silver_base
from talaria.silver_base.step_log import step_log
from talaria.silver_facilities import build_silver_facilities
from talaria.silver_facilities.flag_log import flag_log
from talaria.silver_stage3 import build_silver_stage3, load_references
from talaria.silver_stage3.rule_counts import rule_counts
from talaria.gold import build_gold
from talaria.gold import columns as gold_columns, scoring

In [0]:
@dp.materialized_view(name="bronze_facilities", comment="01 · raw facilities snapshot, load metadata only")
def bronze_facilities():
    source = SOURCE["facilities"]
    return (
        spark.read.table(source)
        .withColumn("_source_table", F.lit(source))
        .withColumn("_ingested_at", F.current_timestamp())
    )


@dp.materialized_view(name="bronze_pincodes", comment="01 · raw pincode directory snapshot")
def bronze_pincodes():
    source = SOURCE["pincodes"]
    return (
        spark.read.table(source)
        .withColumn("_source_table", F.lit(source))
        .withColumn("_ingested_at", F.current_timestamp())
    )


@dp.materialized_view(name="bronze_nfhs5", comment="01 · raw NFHS-5 indicators snapshot")
def bronze_nfhs5():
    source = SOURCE["nfhs5"]
    return (
        spark.read.table(source)
        .withColumn("_source_table", F.lit(source))
        .withColumn("_ingested_at", F.current_timestamp())
    )

In [0]:
@dp.materialized_view(
    name="silver_facilities_base",
    comment="02 · objective cleaning: null text, spelling, formats, duplicates, list cleanup",
)
@dp.expect("unique_id_not_null", "unique_id IS NOT NULL")
def silver_facilities_base():
    return build_silver_base(
        spark.read.table("bronze_facilities"),
        spark.read.table("bronze_pincodes"),
    )

In [0]:
@dp.materialized_view(
    name="silver_facilities",
    comment="03 · values kept as-is, trust flags (groups A–H) added",
)
def silver_facilities():
    run_date = datetime.datetime.now(datetime.timezone.utc).date()
    return build_silver_facilities(
        spark.read.table("silver_facilities_base"),
        spark.read.table("bronze_facilities"),
        spark.read.table("bronze_pincodes"),
        run_date,
    )

In [0]:
@dp.materialized_view(
    name="silver_facilities_stage3",
    comment="05 · rule-based flags (R0, F1, R1-R5, C1-C5), reference-CSV driven",
)
def silver_facilities_stage3():
    return build_silver_stage3(
        spark.read.table("silver_facilities"),
        load_references(),
    )


@dp.materialized_view(
    name="silver_stage3_rule_counts",
    comment="05 · per-rule flagged/remaining row log",
)
def silver_stage3_rule_counts():
    return rule_counts(spark.read.table("silver_facilities_stage3"))


@dp.materialized_view(
    name="silver_base_step_log",
    comment="02 · what Silver base did: per-step counts as rows (section, label, count)",
)
def silver_base_step_log():
    return step_log(
        spark.read.table("bronze_facilities"),
        spark.read.table("silver_facilities_base"),
    )


@dp.materialized_view(
    name="silver_facilities_flag_log",
    comment="03 · what Silver facilities flagged: per-flag counts as rows (group, flag, flagged)",
)
def silver_facilities_flag_log():
    return flag_log(spark.read.table("silver_facilities"))

In [0]:
import sys

sys.path.insert(0, "/Workspace/Users/cyhun18@gmail.com/talaria")

from pyspark import pipelines as dp
from pyspark.sql import functions as F

from talaria.gold import build_gold
from talaria.gold import columns as gold_columns, scoring


def _gold_tables():
    facilities, items = build_gold(spark.read.table("silver_facilities_stage3"))
    return facilities, items


@dp.materialized_view(
    name="gold_facilities",
    comment="06 · one row per Facility: confidence_score (0-1), its log, citations, location_uncertain",
)
def gold_facilities():
    return _gold_tables()[0]


@dp.materialized_view(
    name="gold_facility_items",
    comment="06 · one row per claim: specialty, procedure, equipment, capability, description sentence",
)
def gold_facility_items():
    return _gold_tables()[1]


@dp.materialized_view(
    name="gold_score_distribution",
    comment="06 · facilities per confidence_score value",
)
def gold_score_distribution():
    return spark.read.table("gold_facilities").groupBy("confidence_score").agg(
        F.count("*").alias("facilities")
    )


@dp.materialized_view(
    name="gold_rule_fire_log",
    comment="06 · per scoring rule: tier, reason, facilities where it fired",
)
def gold_rule_fire_log():
    silver = gold_columns.in_scope(spark.read.table("silver_facilities_stage3"))
    counts = silver.agg(
        *[F.count(F.when(rule.fired(), 1)).alias(rule.name) for rule in scoring.RULES]
    )
    rows = F.array(
        *[
            F.struct(
                F.lit(rule.tier).alias("tier"),
                F.lit(rule.name).alias("rule"),
                F.lit(rule.reason).alias("reason"),
                F.col(rule.name).alias("fired"),
            )
            for rule in scoring.RULES
        ]
    )
    return counts.select(F.explode(rows).alias("r")).select("r.*")


@dp.materialized_view(
    name="gold_worst_tier_log",
    comment="06 · facilities per worst tier present (the tier that caps the score)",
)
def gold_worst_tier_log():
    silver = gold_columns.in_scope(spark.read.table("silver_facilities_stage3"))
    worst = F.lit("none")
    for tier in reversed([t for t in scoring.TIERS if t.cap is not None]):
        any_fired = F.lit(False)
        for rule in scoring.rules_in(tier.name):
            any_fired = any_fired | rule.fired()
        worst = F.when(any_fired, F.lit(tier.name)).otherwise(worst)
    tiers = spark.createDataFrame(
        [(t.name,) for t in scoring.TIERS if t.cap is not None] + [("none",)],
        schema=["tier"],
    )
    return tiers.join(
        silver.groupBy(worst.alias("tier")).agg(F.count("*").alias("facilities")),
        "tier",
        "left",
    ).select("tier", F.coalesce("facilities", F.lit(0)).alias("facilities"))